# 02b_tune_paper_protocol: Step 2b - PSO / GA / HHO tuning under the exact Lyu et al. (2025) protocol.

```
office-1, first 12 weekdays, 5-min, outlier smoothing, 7 paper features,
random 70/15/15 row split (AutoGluon-Tabular default). Fitness = validation RMSE;
the test rows are only used for the final report, comparable to the paper's Table 1.
Outputs -> outputs/surrogate/paper_protocol_tuned.csv
Script asli: `scripts/02b_tune_paper_protocol.py` [--budget 100] [--seeds 2] — parameter kini diatur di sel `args` di bawah.
```

Dikonversi dari `scripts/02b_tune_paper_protocol.py`. Jalankan dari folder `notebooks/`.

In [1]:
from types import SimpleNamespace
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.data import FEATURES_PAPER, five_minute_dataset
from obep_mlpaper.metaheuristics import ALGORITHMS
from obep_mlpaper.surrogate import XGB_SPACE, make_xgb, metrics, model_zoo

# Parameters (were command-line arguments in the script; edit here)
args = SimpleNamespace(budget=100, seeds=2, pop=10)
out = ROOT / "outputs" / "surrogate"

d = five_minute_dataset(n_days=12, smooth=True)
idx = np.random.RandomState(0).permutation(len(d))
n = len(idx)
itr, iva, ite = idx[:int(.7 * n)], idx[int(.7 * n):int(.85 * n)], idx[int(.85 * n):]
X, y = d[FEATURES_PAPER].to_numpy(), d["target_kwh"].to_numpy()

In [2]:
def fitness(params):
    m = make_xgb(params).fit(X[itr], y[itr])
    return float(np.sqrt(np.mean((y[iva] - m.predict(X[iva])) ** 2)))

In [3]:
rows = []
m = model_zoo()["XGBoost"].fit(X[itr], y[itr])
rows.append(dict(model="XGBoost (default)", val_RMSE=float(np.sqrt(np.mean((y[iva] - m.predict(X[iva])) ** 2))),
                 **metrics(y[ite], np.clip(m.predict(X[ite]), 0, None))))
for algo, fn in ALGORITHMS.items():
    for seed in range(args.seeds):
        kw = {"PSO": "n_particles", "GA": "pop_size", "HHO": "n_hawks"}[algo]
        r = fn(fitness, XGB_SPACE, budget=args.budget, seed=seed, **{kw: args.pop})
        m = make_xgb(r["best_params"]).fit(X[itr], y[itr])
        rows.append(dict(model=f"XGBoost-{algo}", seed=seed, val_RMSE=r["best_f"],
                         **metrics(y[ite], np.clip(m.predict(X[ite]), 0, None)), params=json.dumps(r["best_params"])))
        print(rows[-1]["model"], seed, f"val RMSE={r['best_f']:.4f} test R2={rows[-1]['R2']:.4f}", flush=True)
rows.append(dict(model="Lyu et al. 2025 WeightedEnsemble_L2", R2=0.933, MAPE_pct=26.094))
rows.append(dict(model="Lyu et al. 2025 LSTM", R2=0.921, MAPE_pct=19.358))
res = pd.DataFrame(rows)
res.to_csv(out / "paper_protocol_tuned.csv", index=False)
print(res.groupby("model", sort=False)[["val_RMSE", "R2", "MAPE_pct", "CVRMSE_pct"]].mean().round(4).to_string())

XGBoost-PSO 0 val RMSE=0.3267 test R2=0.9487


XGBoost-PSO 1 val RMSE=0.3191 test R2=0.9504


XGBoost-GA 0 val RMSE=0.3356 test R2=0.9447


XGBoost-GA 1 val RMSE=0.3383 test R2=0.9434


XGBoost-HHO 0 val RMSE=0.3312 test R2=0.9469


XGBoost-HHO 1 val RMSE=0.3559 test R2=0.9435


                                     val_RMSE      R2  MAPE_pct  CVRMSE_pct
model                                                                      
XGBoost (default)                      0.3989  0.9397   11.9842     28.5157
XGBoost-PSO                            0.3229  0.9495   11.2006     26.0779
XGBoost-GA                             0.3370  0.9441   11.7054     27.4535
XGBoost-HHO                            0.3435  0.9452   11.5772     27.1724
Lyu et al. 2025 WeightedEnsemble_L2       NaN  0.9330   26.0940         NaN
Lyu et al. 2025 LSTM                      NaN  0.9210   19.3580         NaN
